# Your First Generative Program: Structured Sentiment Classifier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibm-granite-community/mellea-cookbook/blob/main/recipes/SentimentClassifier/SentimentClassifier.ipynb)

This is the simplest useful generative program you can write with Mellea.

We will build a **customer review sentiment classifier** for laptop reviews that returns a structured result: label (`positive`, `negative`, or `neutral`) plus a confidence score. Along the way you will see exactly what Mellea adds over a plain LLM call, and why that matters even for a short program.

**What you will learn:**

- How a raw LiteLLM call is constructed — prompt, parameters, and response shape
- Why raw LLM responses are unreliable for structured output
- How `instruct(format=...)` gives you a validated, typed Pydantic object
- How to classify a batch of reviews in a loop

**Prerequisites:**

- Python 3.11 or 3.12
- [Ollama](https://ollama.com/) installed locally with `granite4.1:3b` pulled

Pull the model before running:

```bash
ollama pull granite4.1:3b
```

## Step 1. Set up your environment

You can run this notebook in [Colab](https://colab.research.google.com/), or download it and run locally.

To avoid Python package dependency conflicts, we recommend setting up a [virtual environment](https://docs.python.org/3/library/venv.html).

## Step 2. Set up a backend

### Option A - Local Ollama (no credentials required)

Run this cell to install Ollama inside Colab and pull the IBM Granite model. Skip to *Option B* if you prefer watsonx.ai.

In [ ]:
# Install Ollama
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Start the Ollama daemon in the background
import subprocess
import time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# Pull the IBM Granite model
!ollama pull granite4.1:3b

### Option B - IBM watsonx.ai

See [Getting Started with IBM watsonx](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_WatsonX.ipynb) for setup instructions.

You will need `WATSONX_URL`, `WATSONX_APIKEY`, and `WATSONX_PROJECT_ID`.

In [ ]:
# Uncomment if using watsonx.ai
# from ibm_granite_community.notebook_utils import get_env_var
# get_env_var("WATSONX_APIKEY")
# get_env_var("WATSONX_PROJECT_ID")
# get_env_var("WATSONX_URL")

## Step 3. Install relevant libraries

We will need `mellea`, `litellm` (for both the raw call and the Mellea backend), and `pydantic`.

In [ ]:
%pip install -q git+https://github.com/ibm-granite-community/utils
%pip install -q "mellea[litellm]" litellm pydantic

## Step 4. Import libraries and set constants

We import everything we need and pin the model name in one place so both the raw and Mellea approaches always use the same backend.

In [ ]:
import json
import re
from typing import Literal

import litellm
from litellm import completion
from mellea import start_session, MelleaSession
from mellea.backends.types import ModelOption
from pydantic import BaseModel

# Single model constant shared by both the raw LiteLLM call and the Mellea session.
# Change this one line to switch backends for every step below.
OLLAMA_MODEL = "ollama/ibm/granite4:micro"

## Step 5. Start a Mellea session

A `MelleaSession` wraps the model and provides the `instruct()` API used later. We create it once here and reuse it for every Mellea call.

In [ ]:
# Default: Ollama + Granite locally.
m: MelleaSession = start_session(
    backend_name="litellm",
    model_id=OLLAMA_MODEL,
)

# Uncomment for watsonx.ai:
# m = start_session(
#     backend_name="litellm",
#     model_id="watsonx/ibm/granite-4-h-small",
# )

print("Session ready:", m)

## Step 6. Define the output schema

Before calling the model, declare exactly what a valid response looks like. This Pydantic model becomes the contract that Mellea enforces — and the target shape we manually try to parse in the raw approach.

- `label` is constrained to exactly three values via `Literal` — the model cannot return anything else.
- `confidence` is a float between 0 and 1.
- `reason` is a one-sentence explanation.

In [ ]:
class SentimentResult(BaseModel):
    label: Literal["positive", "negative", "neutral"]
    confidence: float   # 0.0 - 1.0
    reason: str         # one-sentence explanation

# Preview the JSON schema the model is asked to conform to
import json as _json
print("Schema sent to model:")
print(_json.dumps(SentimentResult.model_json_schema(), indent=2))

## Step 7. Define test inputs

We use two carefully chosen reviews throughout the notebook:

- **`clear_review`** — unambiguously positive; both approaches should agree.
- **`ambiguous_review`** — hedged and lukewarm; exposes differences in how each approach handles uncertainty.

In [ ]:
clear_review = "The keyboard feels premium and the battery lasts all day. Absolutely worth the price."
ambiguous_review = "It arrived on time I guess. Works fine most of the time. Not sure if I'd buy again."

print("Clear review:    ", clear_review)
print("Ambiguous review:", ambiguous_review)

## Step 8.1. Raw LLM approach — define the function

This function calls Ollama directly via LiteLLM. There are three manual steps inside that Mellea will later do automatically:

1. **Build the prompt** — embed the review and describe the expected JSON shape in plain text.
2. **Call the model** — `completion()` sends a chat-format message to the model. `response_format={"type": "json_object"}` hints to the model to return JSON, but does not validate it.
3. **Parse the response** — extract the text from `response.choices[0].message.content`, strip any markdown fences with `_extract_json()`, then `json.loads()` it into a plain `dict`.

Notice what is **not** happening: there is no type checking, no field validation, and no guarantee the keys exist.

In [ ]:
def _extract_json(text: str) -> str:
    """Strip markdown code fences if the model wrapped its JSON in them."""
    match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    return match.group(1) if match else text.strip()


def classify_sentiment_raw(review: str) -> dict | None:
    # --- Step 1: build the prompt ---
    # We describe the desired JSON shape in natural language.
    # The model may or may not follow this exactly.
    prompt = f"""Classify the sentiment of this customer review.

Return ONLY valid JSON with this structure:
{{
  "label": "positive" | "negative" | "neutral",
  "confidence": 0.0,
  "reason": "string"
}}

Review: {review}
"""
    try:
        # --- Step 2: call the model ---
        # completion() is a standard LiteLLM call.
        # - model: the Ollama model identifier
        # - messages: chat-format list; we send a single user turn
        # - temperature=0: deterministic output
        # - response_format: hints JSON mode, but does NOT validate the schema
        response: litellm.ModelResponse = completion(  # type: ignore[assignment]
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"},
        )

        # --- Step 3: extract text and parse ---
        # The model response is a ModelResponse object.
        # The actual text lives at response.choices[0].message.content
        content: str = response.choices[0].message.content or ""
        response_text = _extract_json(content)   # strip ```json ... ``` if present
        return json.loads(response_text)          # parse to dict — may raise JSONDecodeError

    except json.JSONDecodeError as error:
        print(f"JSON parsing failed: {error}")
        return None
    except Exception as error:
        print(f"Raw extraction failed: {error}")
        return None

## Step 8.2. Raw LLM — classify the clear review

Run the raw function on the unambiguous positive review. We inspect the raw response object before parsing so you can see exactly what the model returned.

In [ ]:
# Build the prompt independently so we can print it before calling the model
prompt_clear = f"""Classify the sentiment of this customer review.

Return ONLY valid JSON with this structure:
{{
  "label": "positive" | "negative" | "neutral",
  "confidence": 0.0,
  "reason": "string"
}}

Review: {clear_review}
"""

print("=== PROMPT SENT TO MODEL ===")
print(prompt_clear)

# Call the model directly so we can inspect the raw response object
raw_response_clear: litellm.ModelResponse = completion(  # type: ignore[assignment]
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": prompt_clear}],
    temperature=0,
    response_format={"type": "json_object"},
)

print("=== RAW RESPONSE OBJECT ===")
print(f"Type:          {type(raw_response_clear)}")
print(f"Model:         {raw_response_clear.model}")
print(f"Finish reason: {raw_response_clear.choices[0].finish_reason}")
print(f"Raw content:   {raw_response_clear.choices[0].message.content!r}")

In [ ]:
# Now parse with our helper and inspect the result
raw_result_clear = classify_sentiment_raw(clear_review)

print("=== PARSED RESULT ===")
if raw_result_clear:
    print(f"Result type:  {type(raw_result_clear)}")
    print(f"Label:        {raw_result_clear.get('label')}")
    print(f"Confidence:   {raw_result_clear.get('confidence')}  ← raw value, type: {type(raw_result_clear.get('confidence')).__name__}")
    print(f"Reason:       {raw_result_clear.get('reason')}")
    print()
    print("Notice: the result is an unvalidated dict.")
    print("  - 'label' could be 'Positive', 'POSITIVE', or 'very positive'")
    print("  - 'confidence' could be a float, a string, or '95%'")
    print("  - Any key could be missing — .get() returns None silently")
else:
    print("Extraction failed")

## Step 8.3. Raw LLM — classify the ambiguous review

The same raw function on the hedged review. Pay attention to the confidence value — you may see a different type or format than the clear case.

In [ ]:
prompt_ambiguous = f"""Classify the sentiment of this customer review.

Return ONLY valid JSON with this structure:
{{
  "label": "positive" | "negative" | "neutral",
  "confidence": 0.0,
  "reason": "string"
}}

Review: {ambiguous_review}
"""

raw_response_ambiguous: litellm.ModelResponse = completion(  # type: ignore[assignment]
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": prompt_ambiguous}],
    temperature=0,
    response_format={"type": "json_object"},
)

print("=== RAW RESPONSE OBJECT ===")
print(f"Finish reason: {raw_response_ambiguous.choices[0].finish_reason}")
print(f"Raw content:   {raw_response_ambiguous.choices[0].message.content!r}")

In [ ]:
raw_result_ambiguous = classify_sentiment_raw(ambiguous_review)

print("=== PARSED RESULT ===")
if raw_result_ambiguous:
    print(f"Result type:  {type(raw_result_ambiguous)}")
    print(f"Label:        {raw_result_ambiguous.get('label')}  (could be any casing or phrasing)")
    print(f"Confidence:   {raw_result_ambiguous.get('confidence')}  (could be float, string, or percent)")
    print(f"Reason:       {raw_result_ambiguous.get('reason')}")
else:
    print("Extraction failed")

## Step 9.1. Mellea approach — define the function

Now replace all three manual steps with a single `m.instruct()` call. Compare what Mellea does versus what we did by hand:

| Step | Raw approach | Mellea |
|------|-------------|--------|
| Build prompt | Wrote an f-string with JSON instructions | Template string with `{{review}}` placeholder |
| Call model | `litellm.completion()` + parameters | Handled internally by the session |
| Parse response | `json.loads()` → untyped `dict` | Validated against `SentimentResult` → typed object |

The `ModelOption` enum provides backend-agnostic inference options. Using `ModelOption` keys ensures the same options work across all backends (Ollama, OpenAI, watsonx, etc.), making your code portable.

In [ ]:
def classify_sentiment_mellea(review: str) -> SentimentResult:
    # m.instruct() handles prompt construction, model call, JSON parsing,
    # and schema validation in one step.
    #
    # - The template string uses {{review}} as a placeholder.
    # - user_variables substitutes the actual review text safely.
    # - format=SentimentResult tells Mellea to validate the response
    #   against the Pydantic schema before returning.
    # - ModelOption.SEED sets a deterministic seed via the backend-agnostic key.
    result = m.instruct(
        "Classify the sentiment of this customer review. "
        "Provide a label (positive, negative, or neutral), "
        "a confidence score between 0 and 1, "
        "and a one-sentence reason: {{review}}",
        user_variables={"review": review},
        format=SentimentResult,
        model_options={ModelOption.SEED: 42},
    )
    # result.value is the validated JSON string; parse it into the typed model
    assert result.value is not None, "instruct() returned no value"
    return SentimentResult.model_validate_json(result.value)

## Step 9.2. Mellea — classify the clear review

Run the Mellea function on the same clear review used in Step 8a. The return value is a `SentimentResult` instance — not a dict. Notice the field types are guaranteed.

In [ ]:
mellea_result_clear = classify_sentiment_mellea(clear_review)

print("=== MELLEA RESULT ===")
print(f"Result type:     {type(mellea_result_clear)}")
print(f"Label:           {mellea_result_clear.label}")
print(f"Label type:      {type(mellea_result_clear.label).__name__}  ← always str, always one of the three literals")
print(f"Confidence:      {mellea_result_clear.confidence:.0%}")
print(f"Confidence type: {type(mellea_result_clear.confidence).__name__}  ← always float, arithmetic is safe")
print(f"Reason:          {mellea_result_clear.reason}")

## Step 9.3. Mellea — classify the ambiguous review

Run Mellea on the same ambiguous review used in Step 8.3. Compare the label and confidence to the raw result — Mellea may surface a different interpretation, but the types are always guaranteed.

In [ ]:
mellea_result_ambiguous = classify_sentiment_mellea(ambiguous_review)

print("=== MELLEA RESULT ===")
print(f"Result type:     {type(mellea_result_ambiguous)}")
print(f"Label:           {mellea_result_ambiguous.label}  (guaranteed: positive | negative | neutral)")
print(f"Confidence:      {mellea_result_ambiguous.confidence:.0%}  (guaranteed float)")
print(f"Reason:          {mellea_result_ambiguous.reason}")

## Step 9.4. Side-by-side comparison

Look at both results together for each review before moving to batch processing.

| | Raw LLM | Mellea |
|---|---|---|
| **Return type** | `dict` | `SentimentResult` (Pydantic model) |
| **Field access** | `result.get('label')` — no guarantee key exists | `result.label` — IDE autocomplete, type-checked |
| **Label values** | Any string the model produces | Exactly `positive`, `negative`, or `neutral` |
| **Confidence** | Could be float, string, or percent | Always a `float` — arithmetic is safe |
| **Parse failures** | Silent `None` return | `ValidationError` with field-level detail |
| **Downstream code** | Guards everywhere (`if result and 'label' in result`) | Trust the object — `result.label` always exists |

In [ ]:
def _fmt_raw(r: dict | None, field: str) -> str:
    return str(r.get(field)) if r else "ERROR"

for label, review, raw, mellea in [
    ("Clear",     clear_review,     raw_result_clear,     mellea_result_clear),
    ("Ambiguous", ambiguous_review, raw_result_ambiguous, mellea_result_ambiguous),
]:
    print(f"{'─' * 60}")
    print(f"Review ({label}): {review[:60]}...")
    print(f"  Raw    → label={_fmt_raw(raw, 'label')!r:12}  confidence={_fmt_raw(raw, 'confidence')} (raw value)")
    print(f"  Mellea → label={mellea.label!r:12}  confidence={mellea.confidence:.0%} (typed float)")

## Step 10.1. Batch processing — raw LLM

Scale the raw approach to five reviews. Because each result is an untyped dict (or `None` on failure), every access needs a guard.

In [ ]:
reviews = [
    "The keyboard feels premium and the battery lasts all day. Absolutely worth the price.",
    "Stopped working after two weeks. Customer support was unhelpful and slow to respond.",
    "Decent product. Does what it says, nothing more. Packaging could be better.",
    "Incredible build quality. Fastest laptop I have ever owned.",
    "Screen flickering issue from day one. Very disappointed.",
]

In [ ]:
raw_results: list[dict | None] = [classify_sentiment_raw(r) for r in reviews]

print("RAW LLM BATCH")
print(f"{'#':<3} {'Label':<12} {'Conf (raw)':>12}  Review")
print("-" * 72)
for i, (rev, raw) in enumerate(zip(reviews, raw_results), 1):
    if raw:
        # confidence is printed as-is — it may be a float, string, or percent
        # depending on what the model returned. Aggregating it would require
        # extra parsing and type coercion.
        label = str(raw.get("label", "MISSING"))
        conf  = str(raw.get("confidence", "MISSING"))
        print(f"{i:<3} {label:<12} {conf:>12}  {rev[:44]}...")
    else:
        print(f"{i:<3} {'ERROR':<12} {'N/A':>12}  {rev[:44]}...")

print()
print("Note: aggregating 'confidence' across rows is not safe here — the type is unknown.")

## Step 10.2. Batch processing — Mellea

The same five reviews with Mellea. Because every result is a validated `SentimentResult`, confidence is always a `float` and aggregation requires no extra parsing or type guards.

In [ ]:
results: list[SentimentResult] = [classify_sentiment_mellea(r) for r in reviews]

print("MELLEA BATCH")
print(f"{'#':<3} {'Label':<12} {'Conf':>6}  Review")
print("-" * 72)
for i, (rev, res) in enumerate(zip(reviews, results), 1):
    # confidence is a guaranteed float — .0% formatting is always safe
    print(f"{i:<3} {res.label:<12} {res.confidence:>5.0%}  {rev[:44]}...")

print()
# Safe aggregation — no guards needed because the types are enforced by the schema
avg_conf  = sum(r.confidence for r in results) / len(results)
positives = sum(1 for r in results if r.label == "positive")
negatives = sum(1 for r in results if r.label == "negative")
neutrals  = sum(1 for r in results if r.label == "neutral")
print(f"Positive: {positives}  Negative: {negatives}  Neutral: {neutrals}  Avg confidence: {avg_conf:.0%}")

## Key takeaways

- A raw LiteLLM call involves three manual steps: build the prompt, call the model, parse the response. Each step is a potential failure point.
- `response.choices[0].message.content` is always a string — you own all parsing and validation after that.
- Mellea's `instruct(format=...)` collapses those three steps into one and returns a validated typed object.
- The `format=` parameter enforces the schema on every call — no extra code needed in the caller.
- `ModelOption` keys ensure the same configuration works across all backends.

The key shift: instead of parsing what the model returned, you declared what you need — and Mellea made the model conform to it.

**Next steps:**

- [Instruct, Validate, Repair](https://github.com/ibm-granite-community/mellea-cookbook/blob/main/recipes/InstructValidateRepair/InstructValidateRepair.ipynb): Add `Requirement` objects and automatic repair loops to enforce business rules beyond schema validation.
- [Structured Data Extraction](https://github.com/ibm-granite-community/mellea-cookbook/blob/main/recipes/StructuredDataExtraction/StructuredDataExtraction.ipynb): Apply the same patterns to invoices, emails, and support tickets with `@generative` stubs.